# HARE-S0-1 on Kaggle: does System 0 let Qwen3.5-4B catch the 35B?

**Before running:** Settings → Accelerator **GPU T4 x2**, Internet **On** (needs a phone-verified account).
Then **Save Version → Save & Run All (Commit)** so it runs unattended; you can close the tab.

What it does: clones `erpzz/haven` at `53387e7`, unpacks the System 0 harness embedded below (hash-checked against its preregistration),
builds llama.cpp `b11146` for the T4, downloads the pinned Qwen3.5-4B GGUF (sha256-checked), and runs four arms:
4B plain / 4B + System 0 on the frozen 180 episodes, then the same pair on 179 fresh-seed episodes.
Set `RUN_35B = True` to add the 35B pair on the same hardware (22 GB download, about 1 extra hour).

Output: `/kaggle/working/hare-s0/results.json` (sanitized verdict) plus per-arm summaries. The `*.jsonl` files hold full synthetic transcripts. Keep the notebook private.
Synthetic data only, no credentials, no paid services. Expected GPU time: about 2 h with the 4B only, about 3–3.5 h with the 35B.

In [ ]:
RUN_35B = False        # True: also run Qwen3.6-35B-A3B plain + System 0 on the same T4s
SKIP_FRESH = False     # True: frozen set only (saves ~40 min)
HAVEN_COMMIT = "53387e74a0ad90786ac3ea16753c8ad99142e9df"
LLAMA_TAG = "b11146"
WORK = "/kaggle/working/hare-s0"
MODELS = "/tmp/hare-models"   # outside /kaggle/working (20 GB cap)


In [ ]:
import subprocess, os, sys, shutil, json, hashlib, base64, io, tarfile, pathlib, time
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout or "")[-3000:], (r.stderr or "")[-3000:], flush=True)
    if check and r.returncode: raise SystemExit(f"failed: {cmd}")
    return r.stdout
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
sh("nvcc --version | tail -2")
sh("nproc; free -g | head -2; df -h /tmp /kaggle/working | tail -2")
os.makedirs(WORK, exist_ok=True); os.makedirs(MODELS, exist_ok=True)


In [ ]:
# Haven at the pinned commit, plus the embedded System 0 harness
if not os.path.isdir("/tmp/haven/.git"):
    sh("git clone -q --filter=blob:none https://github.com/erpzz/haven /tmp/haven")
sh(f"git -C /tmp/haven checkout -q {HAVEN_COMMIT} && git -C /tmp/haven log -1 --format='%h %s'")
BLOB = "..."
raw = base64.b64decode(BLOB)
assert hashlib.sha256(raw).hexdigest() == "d587a6895b8c730368d8d5f20dee594cf8ea332b7937338f7a735d0bb81ba79e", "embedded harness corrupted"
dest = pathlib.Path("/tmp/haven/labs/intelligence-bakeoff")
shutil.rmtree(dest / "system0", ignore_errors=True)
tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz").extractall(dest)
S0 = dest / "system0"
sh(f"cd {dest} && python3 -m unittest discover -s system0/tests 2>&1 | tail -3")
sh(f"cd {dest} && python3 system0/bench_s0.py check")


In [ ]:
# llama.cpp b11146 with CUDA for the T4 (sm_75); web UI off (its pinned asset download is gone)
ENGINE = "/tmp/llama.cpp/build/bin/llama-server"
if not os.path.exists(ENGINE):
    sh(f"git clone -q --depth 1 --branch {LLAMA_TAG} https://github.com/ggml-org/llama.cpp /tmp/llama.cpp")
    t0 = time.time()
    sh("cd /tmp/llama.cpp && cmake -S . -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 "
       "-DLLAMA_CURL=OFF -DLLAMA_USE_PREBUILT_UI=OFF -DLLAMA_BUILD_UI=OFF "
       "-DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF > /tmp/cmake.log 2>&1 || (tail -40 /tmp/cmake.log; false)")
    sh("cd /tmp/llama.cpp && cmake --build build --target llama-server -j4 > /tmp/build.log 2>&1 || (tail -60 /tmp/build.log; false)")
    print(f"build minutes: {(time.time()-t0)/60:.1f}")
sh(f"{ENGINE} --version 2>&1 | tail -2")


In [ ]:
# Pinned GGUFs, checked against the preregistration
reg = json.loads((S0 / "PREREGISTRATION_S0.json").read_text())
def fetch(spec):
    path = f"{MODELS}/{spec['file']}"
    if not os.path.exists(path):
        url = f"https://huggingface.co/{spec['repo']}/resolve/{spec['revision']}/{spec['file']}"
        sh(f"curl -sSL --retry 5 -o {path}.part '{url}' && mv {path}.part {path}")
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(1 << 24), b""): h.update(c)
    assert h.hexdigest() == spec["sha256"], f"sha256 mismatch for {spec['file']}"
    return path
M4 = fetch(reg["settings"]["4b"])
M35 = None
if RUN_35B:
    free_gb = shutil.disk_usage(MODELS).free / 1e9
    if free_gb < 26: print(f"Only {free_gb:.0f} GB free: skipping the 35B")
    else: M35 = fetch(reg["settings"]["35b"])
print(M4, M35)


In [ ]:
# Run every arm (resumes if interrupted), then the preregistered verdict
gpus = sh("nvidia-smi --query-gpu=name --format=csv,noheader").strip().splitlines()
hw = f"Kaggle {len(gpus)}x {gpus[0]}; 4B on GPU0; 35B split across all GPUs (Slink used --cpu-moe)"
cmd = (f"cd {S0} && python3 run_arms.py --engine {ENGINE} --model-4b {M4} --out {WORK} --hardware '{hw}'"
       + (f" --model-35b {M35}" if M35 else "") + (" --skip-fresh" if SKIP_FRESH else ""))
print("$", cmd, flush=True)
p = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in p.stdout: print(line, end="", flush=True)
assert p.wait() == 0, "run_arms failed; see output above and server-*.log in WORK"


In [ ]:
res = json.loads(pathlib.Path(WORK, "results.json").read_text())
print(json.dumps(res["verdict"], indent=2))
fz = res["comparisons"]["frozen"]
print(f"\nfrozen 180: plain {fz['a_pass']}  ->  System 0 {fz['b_pass']}   (gained {fz['gained']}, lost {fz['lost']}, p={fz['sign_test_p']})")
print(f"35B reference: {res['verdict']['H2_reference_35b']}   mean s/episode plain {fz['mean_episode_seconds']['a']}  s0 {fz['mean_episode_seconds']['b']}")
for f, v in fz["by_family"].items(): print(f"  {f:22s} {v['a']:2d} -> {v['b']:2d} / {v['n']}")
# Sanitized bundle to download and hand back: summaries + results, no transcripts
sh(f"cd {WORK} && tar czf /kaggle/working/hare-s0-results.tgz results.json *.summary.json")
print("Download /kaggle/working/hare-s0-results.tgz (Output tab) and send it back.")
